# 🧠 Synthetic Instruction Dataset for LLM Fine-Tuning

In this lesson, we are going to generate a synthetic instruction dataset — question-and-answer pairs grounded in real documents, that can be used to fine-tune or evaluate a large language model without exposing the source content.

In this notebook, we will:
1. Generate Q&A pairs from a folder of documents using `DocumentQAGeneration`
2. Inspect and quality-filter the output

## 1. Generate Q&A Pairs

`generate()` accepts a folder path, a single file path, or a PyArrow table as `input_source`. `num_qa_pairs` sets how many pairs to generate **per document**.

In [1]:
#add here the subcription key
subscription_key='add-your-subscription-key' 

In [5]:
from ydata.synthesizers.text.model.qa import DocumentQAGeneration

qa_generator = DocumentQAGeneration(
    backend='openai',
    subscription_key=subscription_key,
    model='gpt-5'
)

print('DocumentQAGeneration ready ✓')

DocumentQAGeneration ready ✓


In [6]:
# Generate Q&A pairs from a folder of documents
# Replace 'assets/documents/' with the path to your own document folder
result = qa_generator.generate(
    input_source='../data/documents/',   # folder, single file, or PyArrow table
    docs_extension='docx',              # 'docx' | 'txt' | 'pdf'
    num_qa_pairs=5,                     # pairs per document
)

print(f'Generated {len(result)} Q&A pairs ✓')

100%|████████████████████████████████████████████| 2/2 [00:00<00:00,  7.23it/s]


INFO: 2026-08-26 18:33:44,905 HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO: 2026-08-26 18:33:48,658 HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
Generated 64 Q&A pairs ✓


In [11]:
print(f'Shape: {result.shape}')
print(f'Columns: {list(result.columns)}')
result

Shape: (64, 4)
Columns: ['question', 'answer', 'difficulty', 'source']


,question,answer,difficulty,source
0,What is the document title?,Exploratory Data Analysis Is a Data Engineerin...,easy,../data/documents/Technical blog article_0b0b7...
1,What is the listed reading time for the document?,14 min read,easy,../data/documents/Technical blog article_0b0b7...
2,What category is the document filed under?,Data Engineering,easy,../data/documents/Technical blog article_0b0b7...
3,Name five tags mentioned in the document.,"EDA, Data Engineering, Data Quality, Data Obse...",medium,../data/documents/Technical blog article_0b0b7...
4,"According to the document, why is EDA critical...",It prevents defects from reaching downstream c...,medium,../data/documents/Technical blog article_0b0b7...
...,...,...,...,...
59,What explicit call to action does the article ...,"Pick the most critical pipeline, run a focused...",medium,../data/documents/Technical blog article_f02c7...
60,"In the quick Spark snippet, what is the purpos...",To inspect skew by identifying top heavy-hitte...,medium,../data/documents/Technical blog article_f02c7...
61,"In the workflow’s data acquisition step, what ...",Profile in place with SQL on the warehouse or ...,easy,../data/documents/Technical blog article_f02c7...
62,What does the workflow advise regarding PII du...,"Scan for PII such as emails, SSNs, and phone f...",easy,../data/documents/Technical blog article_f02c7...


In [12]:
# Distribution of Q&A pairs across source documents
print('Pairs per source document:')
print(result['source'].value_counts().to_string())

Pairs per source document:
source
../data/documents/Technical blog article_f02c74c1-65b9-4deb-bd3c-c447689e896d.docx    34
../data/documents/Technical blog article_0b0b701d-100d-4852-a55e-8c8f3856ad7b.docx    30


In [13]:
# Spot-check a single Q&A pair
sample = result.sample(1).iloc[0]
print(f'Source  : {sample["source"]}')
print(f'Question: {sample["question"]}')
print(f'Answer  : {sample["answer"]}')

Source  : ../data/documents/Technical blog article_0b0b701d-100d-4852-a55e-8c8f3856ad7b.docx
Question: How should overlap between fact and dimension tables be evaluated?
Answer  : Measure overlap of join keys and report orphan rates and expected fan-out.


## 2. Export to JSONL for Fine-Tuning

Instruction fine-tuning pipelines expect data in JSONL format — one JSON object per line. `question` maps to `instruction` and `answer` maps to `response`.

In [17]:
import os
import json 
output_path = 'output/instruction_dataset.jsonl'
os.makedirs('output', exist_ok=True)

with open(output_path, 'w') as f:
    for _, row in result.iterrows():
        record = {
            'instruction': row['question'],
            'response':    row['answer'],
            'source':      row['source'],
        }
        f.write(json.dumps(record) + '\n')

print(f'Exported {len(filtered)} pairs → {output_path}')

NameError: name 'json' is not defined

In [ ]:
# Preview the first three records
with open(output_path) as f:
    for i, line in enumerate(f):
        if i >= 3:
            break
        print(json.dumps(json.loads(line), indent=2))
        print()

## ✅ Summary

In this lesson you:

1. **Configured an LLM backend** — wired an Anthropic or OpenAI key into the environment so `DocumentQAGeneration` can call it internally
2. **Generated Q&A pairs** using `DocumentQAGeneration.generate()`, pointed at a folder of documents with a specified file type and pairs-per-document count
3. **Inspected the output** — a DataFrame with `question`, `answer`, and `source` columns, traceable back to each source file

The key insight: synthetic instruction datasets are only as good as their source documents. Combining `DocumentGenerator` (from the previous video) with `DocumentQAGeneration` gives you a fully synthetic pipeline — generate the documents, then generate the instruction dataset from them.